# Análisis exploratorio de ventas y comportamiento de clientes

**Estudiante:** KELY PACHECO

**Módulo:** Friendly SQL & Python para Analytics

**Diploma:** Diploma Data Analyst

## Objetivo

Analizar el comportamiento de las ventas según producto, categoría, segmento, región y canal, identificando patrones de facturación, descuentos y margen que permitan obtener información útil para la gestión comercial.

## Contexto

FerreMarket Perú es una empresa dedicada a la comercialización de productos de ferretería y construcción a través de diferentes canales de venta.

In [119]:
!pip -q install duckdb plotly

In [120]:
import pandas as pd
import numpy as np
import duckdb
import plotly.express as px
from IPython.display import display

pd.set_option("display.max_columns", None)

In [121]:
df = pd.read_csv(
    "dataset_ventas.csv",
    parse_dates=["Fecha"]
)

print("Dataset cargado correctamente")

Dataset cargado correctamente


In [122]:
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])
df.head()
df.tail()
df.info()
print(df.columns.tolist())

Filas: 3000
Columnas: 14
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   VentaID         3000 non-null   object        
 1   Fecha           3000 non-null   datetime64[ns]
 2   ClienteID       3000 non-null   object        
 3   Segmento        3000 non-null   object        
 4   Producto        3000 non-null   object        
 5   Categoria       3000 non-null   object        
 6   Region          3000 non-null   object        
 7   Canal           3000 non-null   object        
 8   Cantidad        3000 non-null   int64         
 9   PrecioUnitario  3000 non-null   int64         
 10  Descuento       3000 non-null   float64       
 11  VentaNeta       3000 non-null   float64       
 12  Margen          3000 non-null   float64       
 13  MedioPago       3000 non-null   object        
dtypes: datetime64[ns](1), float64(3

In [123]:
calidad = pd.DataFrame({
    "Nulos": df.isna().sum(),
    "Porcentaje": (df.isna().mean() * 100).round(2)
})

display(calidad)

,Nulos,Porcentaje
VentaID,0,0.0
Fecha,0,0.0
ClienteID,0,0.0
Segmento,0,0.0
Producto,0,0.0
Categoria,0,0.0
Region,0,0.0
Canal,0,0.0
Cantidad,0,0.0
PrecioUnitario,0,0.0


In [124]:
print(
    "Duplicados:",
    df.duplicated().sum()
)

Duplicados: 0


In [125]:
for col in df.select_dtypes(include="object").columns:
    print(
        col,
        ":",
        (df[col].astype(str).str.strip() == "").sum()
    )

VentaID : 0
ClienteID : 0
Segmento : 0
Producto : 0
Categoria : 0
Region : 0
Canal : 0
MedioPago : 0


In [126]:
df.describe()

,Fecha,Cantidad,PrecioUnitario,Descuento,VentaNeta,Margen
count,3000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000
mean,2024-12-30 17:08:09.600000256,3.956667,54.494667,0.042673,203.588003,47.585730
min,2024-01-01 00:00:00,1.000000,8.000000,0.000000,7.200000,0.930000
25%,2024-07-01 00:00:00,2.000000,18.000000,0.000000,56.000000,11.627500
50%,2024-12-27 00:00:00,4.000000,32.000000,0.050000,108.640000,24.430000
75%,2025-07-07 00:00:00,6.000000,47.750000,0.050000,212.800000,50.367500
max,2025-12-31 00:00:00,7.000000,260.000000,0.100000,1820.000000,582.680000
std,NaN,2.009843,65.734311,0.032237,290.878497,73.391726


In [127]:
df.describe(include="object")

,VentaID,ClienteID,Segmento,Producto,Categoria,Region,Canal,MedioPago
count,3000,3000,3000,3000,3000,3000,3000,3000
unique,3000,499,3,12,6,6,4,5
top,V03000,C0469,Hogar,Cemento 42.5kg,Herramientas,Arequipa,Tienda,Efectivo
freq,1,18,1351,273,749,535,1279,626


### Variables relevantes

Las principales variables consideradas son Fecha, Producto, Categoría, Segmento, Región, Canal, Cantidad, Descuento, VentaNeta y Margen.

Estas variables permiten analizar el comportamiento comercial desde una perspectiva temporal, geográfica, de producto, cliente y rentabilidad.

In [128]:
kpis = pd.Series({
    "Total de operaciones": len(df),
    "Ventas netas": df["VentaNeta"].sum(),
    "Ticket promedio": df["VentaNeta"].mean(),
    "Margen total": df["Margen"].sum(),
    "Margen promedio": df["Margen"].mean()
})

display(kpis.to_frame("Valor"))

,Valor
Total de operaciones,3000.000000
Ventas netas,610764.010000
Ticket promedio,203.588003
Margen total,142757.190000
Margen promedio,47.585730


In [129]:
ventas_categoria = (
    df.groupby("Categoria")["VentaNeta"]
    .sum()
    .reset_index()
    .sort_values("VentaNeta", ascending=False)
)

fig = px.bar(
    ventas_categoria,
    x="Categoria",
    y="VentaNeta",
    text_auto=".2s",
    title="Ventas netas por categoría"
)

fig.show()

In [130]:
mensual = (
    df.set_index("Fecha")
    .resample("ME")["VentaNeta"]
    .sum()
    .reset_index()
)

fig = px.line(
    mensual,
    x="Fecha",
    y="VentaNeta",
    markers=True,
    title="Evolución mensual de ventas"
)

fig.show()

In [131]:
canal = (
    df.groupby("Canal")["VentaNeta"]
    .sum()
    .reset_index()
)

fig = px.pie(
    canal,
    names="Canal",
    values="VentaNeta",
    hole=0.45,
    title="Participación de ventas por canal"
)

fig.show()

In [132]:
fig = px.histogram(
    df,
    x="VentaNeta",
    nbins=40,
    title="Distribución de ventas netas"
)

fig.show()

In [133]:
# Hallazgo 4: segmento con mayor ticket promedio
ticket_segmento = (
    df.groupby("Segmento")["VentaNeta"]
    .mean()
    .sort_values(ascending=False)
)

segmento_top = ticket_segmento.index[0]
ticket_promedio_top = ticket_segmento.iloc[0]

In [134]:
fig = px.bar(
    ticket_segmento.reset_index(),
    x="Segmento",
    y="VentaNeta",
    text_auto=".2s",
    title="Ticket Promedio por Segmento"
)
fig.update_traces(texttemplate='S/ %{y:,.2f}', textposition='outside')
fig.show()

In [135]:
fig = px.scatter(
    df,
    x="Descuento",
    y="Margen",
    size="VentaNeta",
    color="Categoria",
    hover_data=[
        "Producto",
        "Canal"
    ],
    title="Relación entre descuento y margen"
)

fig.show()

In [136]:
q1 = duckdb.query("""
SELECT *
FROM df
LIMIT 10
""").to_df()

display(q1)

,VentaID,Fecha,ClienteID,Segmento,Producto,Categoria,Region,Canal,Cantidad,PrecioUnitario,Descuento,VentaNeta,Margen,MedioPago
0,V00001,2024-03-03,C0349,Hogar,Martillo profesional,Herramientas,Arequipa,App,6,65,0.03,378.3,116.85,Efectivo
1,V00002,2024-12-31,C0186,Empresa,Brocha 3 pulgadas,Pinturas,Piura,App,6,18,0.00,108.0,23.66,Transferencia
2,V00003,2025-10-11,C0032,Profesional,Foco LED,Electricidad,Junín,App,4,15,0.00,60.0,14.52,Transferencia
3,V00004,2025-05-10,C0390,Hogar,Brocha 3 pulgadas,Pinturas,Cusco,Tienda,3,18,0.10,48.6,15.38,Plin
4,V00005,2024-09-25,C0484,Hogar,Foco LED,Electricidad,La Libertad,Web,2,15,0.05,28.5,8.09,Yape
5,V00006,2024-08-29,C0114,Profesional,Alicate,Herramientas,La Libertad,App,2,38,0.00,76.0,17.07,Plin
6,V00007,2024-11-14,C0403,Hogar,Cable eléctrico 10m,Electricidad,La Libertad,App,5,42,0.03,203.7,61.74,Yape
7,V00008,2025-08-05,C0394,Profesional,Cable eléctrico 10m,Electricidad,Cusco,App,1,42,0.00,42.0,5.11,Transferencia
8,V00009,2024-06-28,C0058,Profesional,Foco LED,Electricidad,Arequipa,Web,4,15,0.05,57.0,8.59,Plin
9,V00010,2024-03-07,C0280,Hogar,Taladro inalámbrico,Herramientas,Arequipa,Web,6,260,0.05,1482.0,358.33,Tarjeta


In [137]:
q2 = duckdb.query("""
SELECT *
FROM df
WHERE Canal = 'Web'
AND VentaNeta > 500
ORDER BY VentaNeta DESC
LIMIT 20
""").to_df()

display(q2)

,VentaID,Fecha,ClienteID,Segmento,Producto,Categoria,Region,Canal,Cantidad,PrecioUnitario,Descuento,VentaNeta,Margen,MedioPago
0,V01928,2025-01-15,C0412,Hogar,Taladro inalámbrico,Herramientas,Junín,Web,7,260,0.00,1820.0,529.86,Transferencia
1,V02296,2024-08-30,C0141,Hogar,Taladro inalámbrico,Herramientas,Arequipa,Web,7,260,0.00,1820.0,550.72,Tarjeta
2,V00368,2025-04-13,C0376,Profesional,Taladro inalámbrico,Herramientas,Cusco,Web,7,260,0.00,1820.0,441.66,Plin
3,V00363,2024-04-14,C0189,Hogar,Taladro inalámbrico,Herramientas,Piura,Web,7,260,0.05,1729.0,504.73,Plin
4,V00296,2025-12-12,C0130,Profesional,Taladro inalámbrico,Herramientas,Piura,Web,7,260,0.05,1729.0,576.88,Efectivo
5,V02918,2024-07-30,C0390,Hogar,Taladro inalámbrico,Herramientas,La Libertad,Web,7,260,0.05,1729.0,400.87,Efectivo
6,V02682,2024-12-11,C0238,Hogar,Taladro inalámbrico,Herramientas,Junín,Web,7,260,0.05,1729.0,538.52,Yape
7,V00267,2025-06-11,C0430,Empresa,Taladro inalámbrico,Herramientas,Junín,Web,7,260,0.05,1729.0,450.55,Yape
8,V01685,2025-12-28,C0313,Profesional,Taladro inalámbrico,Herramientas,La Libertad,Web,6,260,0.00,1560.0,433.00,Tarjeta
9,V00329,2025-11-29,C0365,Empresa,Taladro inalámbrico,Herramientas,Junín,Web,6,260,0.00,1560.0,188.87,Transferencia


In [138]:
q3 = duckdb.query("""
SELECT
    Categoria,
    COUNT(*) AS Operaciones,
    SUM(VentaNeta) AS Ventas,
    AVG(Margen) AS MargenPromedio
FROM df
GROUP BY Categoria
""").to_df()

display(q3)

,Categoria,Operaciones,Ventas,MargenPromedio
0,Herramientas,749,334223.18,105.251482
1,Pinturas,499,105126.30,48.539479
2,Electricidad,475,49747.32,24.323158
3,Seguridad,500,55459.89,25.748220
4,Plomería,504,32451.16,14.972718
5,Construcción,273,33756.16,28.310513


In [139]:
q4 = duckdb.query("""
SELECT
    Region,
    SUM(VentaNeta) AS VentasTotales,
    AVG(VentaNeta) AS TicketPromedio
FROM df
GROUP BY Region
ORDER BY VentasTotales DESC
""").to_df()

display(q4)

,Region,VentasTotales,TicketPromedio
0,Junín,112198.12,228.044959
1,La Libertad,110441.43,215.705918
2,Arequipa,102090.19,190.822785
3,Piura,98406.14,203.739420
4,Lima,97161.58,191.262953
5,Cusco,90466.55,192.482021


In [140]:
q5 = duckdb.query("""
SELECT
    Canal,
    Segmento,
    COUNT(*) AS Operaciones,
    SUM(VentaNeta) AS Ventas,
    AVG(Descuento) AS DescuentoPromedio,
    AVG(Margen) AS MargenPromedio
FROM df
GROUP BY Canal, Segmento
HAVING SUM(VentaNeta) > 100000
ORDER BY Ventas DESC
""").to_df()

display(q5)

,Canal,Segmento,Operaciones,Ventas,DescuentoPromedio,MargenPromedio
0,Tienda,Hogar,577,114306.66,0.041005,46.632912
1,Tienda,Profesional,502,102813.59,0.043904,48.562809


## 5. Hallazgos

### Hallazgo 1 — Categoría con mayores ventas

La categoría **HERRAMIENTAS** registra las mayores ventas netas, alcanzando **S/ 110K**.

### Hallazgo 2 — Canal con mayores ventas

El canal **TIENDA** concentra el mayor volumen de ventas netas, con **S/ *42.3 %*.

### Hallazgo 3 — Región con mayores ventas

La región **JUNIN** presenta el mayor nivel de ventas, alcanzando **S/ 112K**.

### Hallazgo 4 — Segmento con mayor ticket

El segmento **PROFESIONAL** presenta el mayor ticket promedio, con **S/ 118K** por operación.

### Hallazgo 5 — Mayor margen promedio

La categoría **HERRAMIENTAS** presenta el mayor margen promedio, con **S/ 105K** por operación.